<a href="https://colab.research.google.com/github/enesavci16/fl-its-anomaly-detection/blob/main/02_baseline_and_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Baseline ve Etiketleme (Leakage-Free)

**Amaç:**
1. 4 sensör için Z-skoru tabanlı anomali etiketleri üretmek (**veri sızıntısı yok**)
2. IID ve Non-IID veri bölümleme stratejilerini hazırlamak (S2/S3 için)
3. **S1 Senaryosu:** Merkezi Isolation Forest baseline F1 skorunu hesaplamak

---

## Veri Sızıntısı (Data Leakage) Karşıtı Tasarım

Bu notebook, klasik FL/anomali tespit pipeline'larında karşılaşılan **6 sızıntı noktasını** sistematik olarak kapatır:

| # | Sızıntı Noktası | Kapatma Yöntemi |
|---|---|---|
| 1 | Z-skoru `mean`/`std`'nin tüm veriden hesaplanması | **Sadece train'den** hesaplanır, val+test'e uygulanır |
| 2 | Etiketlemeden önce shuffle | Etiketlemeden önce **kronolojik split** yapılır |
| 3 | Hiperparametre tuning'de test seti kullanımı | Tüm tuning **val** üzerinde, test **bir kez** |
| 4 | IID havuzlamada train↔test geçişi | Train/val/test havuzları **ayrı ayrı** karıştırılıp bölünür |
| 5 | Görsellerin test bilgisini açığa vurması | fig07 sadece **train** anomali oranlarını gösterir |
| 6 | Etiket dosyalarının split bilgisi taşımaması | `labels_sensor_XXX.csv`'ye **`split` kolonu** eklenir |

---

**Bağlam:** Notebook 01'de PeMS04'ten P25/P50/P75/P95 stratejisiyle 4 heterojen sensör seçildi.

**Çıktılar:**
- `data/processed/labels_sensor_XXX.csv` × 4 (train + val + test, `split` kolonu ile)
- `data/splits/iid/client_X.csv` × 4 (her clientta train/val/test ayrımı korunur)
- `data/splits/non_iid/client_X.csv` × 4
- `data/processed/zscore_stats.json` (her sensörün train mean/std'si — şeffaflık için)
- `results/s1_baseline_metrics.csv`
- `results/fig07_train_anomaly_rates.png`
- `results/fig08_s1_confusion_matrix.png`
- `results/fig09_contamination_grid.png`


In [5]:
# GitHub'dan repo klonla (her Colab oturumu başında bir kez yapılır)
import os

REPO = "enesavci16/fl-its-anomaly-detection"

if not os.path.exists("/content/fl-its-anomaly-detection"):
    !git clone https://github.com/{REPO}.git /content/fl-its-anomaly-detection

os.chdir("/content/fl-its-anomaly-detection")
print("Çalışma dizini:", os.getcwd())
print("İçerik:", os.listdir("."))

Çalışma dizini: /content/fl-its-anomaly-detection
İçerik: ['.git', 'data', 'README.md', 'results', 'notebooks', '.gitignore', 'src']


## 1. Setup ve Yol Tespiti


In [6]:
# Standart kütüphaneler
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

# Tekrarlanabilirlik
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Split oranları (sensör başına KRONOLOJİK)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15  # = 1 - TRAIN_RATIO - VAL_RATIO
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

FEATURES = ["flow", "speed", "occupancy"]
Z_THRESHOLD = 3.0  # |z| > 3 kuralı

# IEEE-stil görselleştirme (Notebook 01 ile tutarlı)
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

print("Kütüphaneler yüklendi.")
print(f"Split oranları: train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")


Kütüphaneler yüklendi.
Split oranları: train=0.7, val=0.15, test=0.15


In [7]:
# Dinamik yol tespiti (Sorun #6 çözümü — Colab notebooks/ alt-klasör uyumlu)
if os.path.basename(os.getcwd()) == "notebooks":
    DATA_DIR = "../data"
    RESULTS_DIR = "../results"
else:
    DATA_DIR = "data"
    RESULTS_DIR = "results"

PROCESSED_DIR = f"{DATA_DIR}/processed"
SPLITS_DIR = f"{DATA_DIR}/splits"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{SPLITS_DIR}/iid", exist_ok=True)
os.makedirs(f"{SPLITS_DIR}/non_iid", exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"DATA_DIR     = {DATA_DIR}")
print(f"RESULTS_DIR  = {RESULTS_DIR}")


DATA_DIR     = data
RESULTS_DIR  = results


## 2. Seçilen 4 Sensörün Yüklenmesi


In [8]:
summary = pd.read_csv(f"{RESULTS_DIR}/selected_sensors_summary.csv")
print("Seçilen sensör özeti:")
print(summary)

SENSOR_IDS = summary["sensor_id"].tolist()
print(f"\nSensör ID'leri: {SENSOR_IDS}")


FileNotFoundError: [Errno 2] No such file or directory: 'results/selected_sensors_summary.csv'

In [ ]:
sensor_data = {}
for sid in SENSOR_IDS:
    path = f"{PROCESSED_DIR}/sensor_{sid:03d}.csv"
    df = pd.read_csv(path)
    # Zaman sırasının korunduğunu varsayıyoruz (Notebook 01 çıktısı zaten kronolojik)
    sensor_data[sid] = df.reset_index(drop=True)
    print(f"sensor_{sid:03d}.csv: shape={df.shape}")


## 3. Pipeline: ÖNCE Split, SONRA Etiket

**Kritik sıralama** (data leakage karşıtı):

1. **Adım A — Kronolojik split (ham veri):** Her sensör için ilk %70 train, ortadaki %15 val, son %15 test.
2. **Adım B — Sadece train'den `mean`/`std`:** Z-skoru istatistikleri yalnızca train kısmından hesaplanır.
3. **Adım C — Aynı istatistiklerle 3 split'i de etiketle:** `(x - μ_train) / σ_train`, `|z| > 3` ise anomali.

Bu sırayla yaptığımız sürece test seti hakkında hiçbir bilgi train'e veya etiketleme sürecine sızmaz.

**Çok-değişkenli kombinasyon:** OR kuralı — herhangi bir özellik (flow / speed / occupancy) için `|z| > 3` ise zaman adımı anomali.


In [ ]:
def split_chronological(df, train_ratio=0.70, val_ratio=0.15):
    """Sensörün zaman serisini kronolojik olarak böler. ASLA shuffle YAPMAZ."""
    n = len(df)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train = df.iloc[:n_train].copy()
    val = df.iloc[n_train:n_train + n_val].copy()
    test = df.iloc[n_train + n_val:].copy()
    train["split"] = "train"
    val["split"] = "val"
    test["split"] = "test"
    return train, val, test


def compute_train_stats(train_df, features=FEATURES):
    """Z-skoru istatistiklerini SADECE train'den hesaplar."""
    return {
        feat: {
            "mean": float(train_df[feat].mean()),
            "std": float(train_df[feat].std()),
        }
        for feat in features
    }


def label_with_stats(df, stats, threshold=Z_THRESHOLD):
    """Önceden hesaplanmış (train) istatistiklerle etiketle. Test/val'de kullanılır."""
    df = df.copy()
    z_flags = []
    for feat, st in stats.items():
        sigma = st["std"]
        if sigma == 0:
            df[f"z_{feat}"] = 0.0
            z_flags.append(np.zeros(len(df), dtype=bool))
        else:
            df[f"z_{feat}"] = (df[feat] - st["mean"]) / sigma
            z_flags.append(df[f"z_{feat}"].abs() > threshold)
    df["is_anomaly"] = np.any(z_flags, axis=0).astype(int)
    return df


In [ ]:
# Her sensör için: split → train stats → etiketle → kaydet
labeled_data = {}     # {sid: {"train": df, "val": df, "test": df, "stats": dict}}
sensor_stats = {}     # {sid: stats} — şeffaflık için JSON'a yazacağız

for sid in SENSOR_IDS:
    raw = sensor_data[sid]

    # ADIM A: Kronolojik split (etiketsiz ham veri)
    train_raw, val_raw, test_raw = split_chronological(raw, TRAIN_RATIO, VAL_RATIO)

    # ADIM B: Sadece train'den istatistikler
    stats = compute_train_stats(train_raw)
    sensor_stats[sid] = stats

    # ADIM C: Aynı istatistiklerle 3 split'i de etiketle
    train_lab = label_with_stats(train_raw, stats)
    val_lab = label_with_stats(val_raw, stats)
    test_lab = label_with_stats(test_raw, stats)

    labeled_data[sid] = {
        "train": train_lab, "val": val_lab, "test": test_lab,
        "stats": stats,
    }

    # Tek dosyaya birleştir (split kolonu sayesinde sonradan ayrıştırılabilir)
    full = pd.concat([train_lab, val_lab, test_lab], ignore_index=True)
    full.to_csv(f"{PROCESSED_DIR}/labels_sensor_{sid:03d}.csv", index=False)

    print(f"sensor_{sid:03d}: "
          f"train={len(train_lab):>5d} (anom={train_lab['is_anomaly'].mean():.4f})  "
          f"val={len(val_lab):>5d} (anom={val_lab['is_anomaly'].mean():.4f})  "
          f"test={len(test_lab):>5d} (anom={test_lab['is_anomaly'].mean():.4f})")

# Şeffaflık: Train istatistiklerini JSON'a kaydet
with open(f"{PROCESSED_DIR}/zscore_stats.json", "w") as f:
    json.dump({str(k): v for k, v in sensor_stats.items()}, f, indent=2)
print(f"\nKaydedildi: {PROCESSED_DIR}/zscore_stats.json")


**Beklenti:** Train ve val/test anomali oranları **birbirine yakın olabilir veya farklı olabilir**. Farklılık doğaldır çünkü:
- Train kronolojik olarak önce → mevsimsel/saatlik patern bias'ı taşır
- Val/test sonraki dönemler → farklı trafik koşulları görebilir

Bu beklenen davranış, leakage göstergesi değildir.


### fig07 — Sensör başına TRAIN anomali oranları

**Sızıntı önleme:** Bu görselde **sadece train** verisindeki oranlar gösterilir. Val/test oranlarını görselleştirip rapora koyarsak, bildirinin metodoloji bölümü test setini "görmüş" olur.


In [ ]:
# Sadece train üzerinde özellik bazında anomali oranları
detail_rows = []
for sid in SENSOR_IDS:
    train = labeled_data[sid]["train"]
    row = {"sensor_id": f"S{sid:03d}"}
    for feat in FEATURES:
        row[feat] = (train[f"z_{feat}"].abs() > Z_THRESHOLD).mean()
    row["any (OR)"] = train["is_anomaly"].mean()
    detail_rows.append(row)
detail_df = pd.DataFrame(detail_rows).set_index("sensor_id")
print("TRAIN üzerinde anomali oranları:")
print(detail_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.8))
detail_df.plot(kind="bar", ax=ax, width=0.78,
               color=["#4c72b0", "#dd8452", "#55a467", "#c44e52"])
ax.set_ylabel("Anomali Oranı (Train)")
ax.set_xlabel("Sensör")
ax.set_title("Z-Skoru Tabanlı Anomali Oranları — TRAIN seti (|z| > 3)")
ax.legend(title="Özellik", loc="upper left", framealpha=0.9)
ax.set_xticklabels(detail_df.index, rotation=0)
ax.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig07_train_anomaly_rates.png")
plt.show()
print(f"Kaydedildi: {RESULTS_DIR}/fig07_train_anomaly_rates.png")


## 4. IID / Non-IID İstemci Bölümlemesi (S2 / S3 hazırlığı)

**Sızıntı karşıtı kritik tasarım:** IID havuzlama yapılırken train/val/test havuzları **ayrı ayrı** karıştırılıp bölünür. Yani test verisindeki bir örnek **asla başka bir client'ın train'ine geçemez**.

| Strateji | Açıklama | Sonuç |
|---|---|---|
| **Non-IID (S3)** | Her sensör = bir client. Doğal heterojenlik korunur. | Client_i = Sensör_i'nin train+val+test'i |
| **IID (S2)** | Her split (train/val/test) **ayrı** havuzlanır, shuffle edilir, 4'e bölünür. | Client_i = train_pool[i. dilim] + val_pool[i. dilim] + test_pool[i. dilim] |

Hem IID hem Non-IID **aynı temporal split sınırlarını** kullanır → karşılaştırma adildir.


In [ ]:
def pool_split(split_name):
    """Tüm sensörlerin belirli bir split'ini (train/val/test) tek bir havuzda topla."""
    parts = []
    for sid in SENSOR_IDS:
        d = labeled_data[sid][split_name].copy()
        d["sensor_id"] = sid
        parts.append(d)
    return pd.concat(parts, ignore_index=True)


# Üç havuz da AYRI AYRI hazırlanır (sınırlar katı)
train_pool = pool_split("train")
val_pool = pool_split("val")
test_pool = pool_split("test")
print(f"Havuzlar: train={len(train_pool)}, val={len(val_pool)}, test={len(test_pool)}")


In [ ]:
# --- Non-IID bölümleme ---
# Her sensör → bir client. Split kolonu zaten dosyada.
for client_idx, sid in enumerate(SENSOR_IDS):
    train = labeled_data[sid]["train"].copy()
    val = labeled_data[sid]["val"].copy()
    test = labeled_data[sid]["test"].copy()
    for d in (train, val, test):
        d["sensor_id"] = sid
    full = pd.concat([train, val, test], ignore_index=True)
    full.to_csv(f"{SPLITS_DIR}/non_iid/client_{client_idx}.csv", index=False)
    print(f"[Non-IID] client_{client_idx}: sensor={sid}, "
          f"train={len(train)}, val={len(val)}, test={len(test)}, "
          f"train_anom_rate={train['is_anomaly'].mean():.4f}")


In [ ]:
# --- IID bölümleme ---
# KRİTİK: Her split AYRI AYRI shuffle + 4 parçaya bölünür.
# Test örneği başka client'ın train'ine ASLA geçmez.
n_clients = 4

def split_into_chunks(df, n_chunks):
    """DataFrame'i n eşit parçaya böler (iloc tabanlı, pandas/numpy sürüm bağımsız)."""
    n = len(df)
    base = n // n_chunks
    remainder = n % n_chunks
    chunks = []
    start = 0
    for i in range(n_chunks):
        # İlk `remainder` parça 1 fazla satır alır (eşit dağılım)
        size = base + (1 if i < remainder else 0)
        chunks.append(df.iloc[start:start + size].copy())
        start += size
    return chunks

client_parts = {i: [] for i in range(n_clients)}

for split_name, pool_df in [("train", train_pool), ("val", val_pool), ("test", test_pool)]:
    shuffled = pool_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    chunks = split_into_chunks(shuffled, n_clients)
    for i, chunk in enumerate(chunks):
        client_parts[i].append(chunk)

for i in range(n_clients):
    full = pd.concat(client_parts[i], ignore_index=True)
    full.to_csv(f"{SPLITS_DIR}/iid/client_{i}.csv", index=False)
    train_part = full[full["split"] == "train"]
    print(f"[IID]     client_{i}: train={len(train_part)}, "
          f"val={(full['split']=='val').sum()}, test={(full['split']=='test').sum()}, "
          f"train_anom_rate={train_part['is_anomaly'].mean():.4f}")


**Beklenti:**
- **Non-IID** istemcilerinin train anomali oranları birbirinden **belirgin farklı** olmalı (P25 vs P95 sensörleri).
- **IID** istemcilerinin train anomali oranları birbirine **çok yakın** (≈ havuz oranı) olmalı.

Bu iki gözlem doğrulanırsa, S2 vs S3 karşılaştırması anlamlı bir kontrast üretir.


## 5. S1 — Merkezi Isolation Forest Baseline

**Tasarım kararları:**

| Karar | Seçim | Gerekçe |
|---|---|---|
| Mimari | 4 sensörün train+val+test havuzu (centralized) | Tüm verinin tek sunucuda olduğu klasik baseline |
| Özellikler | flow, speed, occupancy (3D) | S1/S2/S3'te aynı feature seti — adil karşılaştırma |
| Eğitim | `IF.fit(X_train)` — y kullanılmaz | IF gözetimsizdir |
| Hiperparametre tuning | `contamination` grid: `{0.5x, 1x, 1.5x, 2x} × train_anom_rate` | Val F1'e göre seçim, **test asla görülmez** |
| Final değerlendirme | Seçilen `contamination` ile **test üzerinde tek seferlik** | Hiperparametre tuning sızıntısı yok |
| `n_estimators` | 100 | Default, makul |
| `random_state` | 42 | Tekrarlanabilirlik |


In [ ]:
# Havuzlanmış verilerden X, y matrisleri
X_train = train_pool[FEATURES].values
y_train = train_pool["is_anomaly"].values  # IF eğitimde KULLANMAZ — sadece contamination grid için

X_val = val_pool[FEATURES].values
y_val = val_pool["is_anomaly"].values

X_test = test_pool[FEATURES].values
y_test = test_pool["is_anomaly"].values

train_anom_rate = float(y_train.mean())
val_anom_rate = float(y_val.mean())
test_anom_rate = float(y_test.mean())

print(f"Train: n={len(X_train):>6d}, anom_rate={train_anom_rate:.4f}")
print(f"Val  : n={len(X_val):>6d}, anom_rate={val_anom_rate:.4f}")
print(f"Test : n={len(X_test):>6d}, anom_rate={test_anom_rate:.4f}")


In [ ]:
# Contamination grid — TRAIN observed rate'in 4 katı
contamination_grid = sorted(set(
    float(np.clip(train_anom_rate * mult, 0.001, 0.5))
    for mult in [0.5, 1.0, 1.5, 2.0]
))
print(f"Contamination grid (train_rate={train_anom_rate:.4f} bazında):")
for c in contamination_grid:
    print(f"  c = {c:.5f}")


In [ ]:
# Grid search SADECE validation üzerinde — test seti DOKUNULMAZ
grid_results = []

for c in contamination_grid:
    iso = IsolationForest(
        n_estimators=100,
        contamination=c,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    iso.fit(X_train)  # y kullanılmıyor
    y_pred_val = (iso.predict(X_val) == -1).astype(int)
    f1_val = f1_score(y_val, y_pred_val, pos_label=1, zero_division=0)
    prec_val = precision_score(y_val, y_pred_val, pos_label=1, zero_division=0)
    rec_val = recall_score(y_val, y_pred_val, pos_label=1, zero_division=0)
    grid_results.append({
        "contamination": c,
        "f1_val": f1_val,
        "precision_val": prec_val,
        "recall_val": rec_val,
        "model": iso,
    })
    print(f"  c={c:.5f}  →  val F1={f1_val:.4f}  P={prec_val:.4f}  R={rec_val:.4f}")

# En iyi modeli seç (val F1 maksimize)
best = max(grid_results, key=lambda r: r["f1_val"])
best_iso = best["model"]
best_c = best["contamination"]
print(f"\nSeçilen contamination = {best_c:.5f}  (val F1 = {best['f1_val']:.4f})")


In [ ]:
# fig09 — Contamination grid taraması (val sonuçları)
grid_df = pd.DataFrame([
    {"contamination": r["contamination"],
     "F1": r["f1_val"], "Precision": r["precision_val"], "Recall": r["recall_val"]}
    for r in grid_results
])

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for col, marker, color in [("F1", "o", "#4c72b0"),
                            ("Precision", "s", "#dd8452"),
                            ("Recall", "^", "#55a467")]:
    ax.plot(grid_df["contamination"], grid_df[col],
            marker=marker, color=color, label=col, linewidth=1.6, markersize=7)

ax.axvline(best_c, color="red", linestyle="--", alpha=0.6,
           label=f"Seçilen c={best_c:.4f}")
ax.set_xlabel("Contamination")
ax.set_ylabel("Skor (Validation)")
ax.set_title("S1 — Contamination Grid Taraması (Val seti üzerinde)")
ax.legend(loc="best", framealpha=0.9)
ax.grid(alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig09_contamination_grid.png")
plt.show()
print(f"Kaydedildi: {RESULTS_DIR}/fig09_contamination_grid.png")


### Test seti — bir kerelik değerlendirme

Bu noktaya kadar test setine **hiç dokunmadık**. Şimdi seçilmiş model üzerinde **tek seferlik** test değerlendirmesi yapıyoruz. Bu sonuç bildiri Tablo 2'de S1 satırı olur.


In [ ]:
# TEST setinde tek seferlik değerlendirme
y_pred_test_raw = best_iso.predict(X_test)            # -1 (anom) / +1 (normal)
y_pred_test = (y_pred_test_raw == -1).astype(int)     # 1 / 0

f1_test = f1_score(y_test, y_pred_test, pos_label=1, zero_division=0)
prec_test = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)
rec_test = recall_score(y_test, y_pred_test, pos_label=1, zero_division=0)
cm_test = confusion_matrix(y_test, y_pred_test, labels=[0, 1])

print("=" * 56)
print("S1 — MERKEZİ ISOLATION FOREST BASELINE — TEST SONUÇLARI")
print("=" * 56)
print(f"Seçilen contamination: {best_c:.5f}")
print(f"Test F1       : {f1_test:.4f}")
print(f"Test Precision: {prec_test:.4f}")
print(f"Test Recall   : {rec_test:.4f}")
print()
print("Confusion matrix [satır=gerçek, sütun=tahmin]:")
print(f"               normal  anomali")
print(f"  normal    [{cm_test[0,0]:>7d} {cm_test[0,1]:>7d}]")
print(f"  anomali   [{cm_test[1,0]:>7d} {cm_test[1,1]:>7d}]")
print()
print("Sınıflandırma raporu:")
print(classification_report(y_test, y_pred_test, target_names=["normal", "anomaly"], zero_division=0))


In [ ]:
# Metrikleri CSV'ye kaydet (bildiri Tablo 2 — S1 satırı)
metrics_row = {
    "scenario": "S1_centralized",
    "model": "IsolationForest",
    "n_train": len(train_pool),
    "n_val": len(val_pool),
    "n_test": len(test_pool),
    "train_anomaly_rate": train_anom_rate,
    "val_anomaly_rate": val_anom_rate,
    "test_anomaly_rate": test_anom_rate,
    "selected_contamination": best_c,
    "val_f1": best["f1_val"],
    "test_f1": f1_test,
    "test_precision": prec_test,
    "test_recall": rec_test,
    "tn": int(cm_test[0, 0]), "fp": int(cm_test[0, 1]),
    "fn": int(cm_test[1, 0]), "tp": int(cm_test[1, 1]),
}
metrics_path = f"{RESULTS_DIR}/s1_baseline_metrics.csv"
pd.DataFrame([metrics_row]).to_csv(metrics_path, index=False)
print(f"Kaydedildi: {metrics_path}")


### fig08 — S1 Test Confusion Matrix


In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.8))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Anomali"],
            yticklabels=["Normal", "Anomali"],
            cbar=False, ax=ax,
            annot_kws={"size": 11})
ax.set_xlabel("Tahmin")
ax.set_ylabel("Gerçek (Z-skoru, train statlarıyla)")
ax.set_title(f"S1 — Merkezi IF (Test F1 = {f1_test:.3f})")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig08_s1_confusion_matrix.png")
plt.show()
print(f"Kaydedildi: {RESULTS_DIR}/fig08_s1_confusion_matrix.png")


## 6. H1 Kapanış Özeti

**Üretilenler:**
- ✅ 4 sensör için Z-skoru etiketleri (`labels_sensor_XXX.csv`, **`split` kolonu ile**)
- ✅ Train Z-skoru istatistikleri (`zscore_stats.json` — şeffaflık)
- ✅ Non-IID bölümleme (`splits/non_iid/client_X.csv`) — S3 için hazır
- ✅ IID bölümleme (`splits/iid/client_X.csv`) — S2 için hazır (temporal sınırlar korunmuş)
- ✅ S1 baseline test F1 → bildiri Tablo 2 / S1 satırı
- ✅ fig07 (train anomali oranları), fig08 (test confusion matrix), fig09 (contamination grid)

**Data leakage karşıtı uygulanmış 6 önlem:**
1. Önce kronolojik split, sonra etiketleme
2. Z-skoru `mean`/`std` sadece train'den
3. Hiperparametre tuning sadece val'da
4. Test 1 kez, en sonda
5. IID havuzlama train/val/test ayrımına saygılı
6. fig07 sadece train rate'lerini gösterir

**Sonraki adım (H2 — 21-27 Nisan):**
Mininet üzerinde 4 sensör + 1 TMC + OpenFlow switch topolojisi kurulumu.

**Yorum kontrol noktaları:**
1. Test F1, val F1'den ne kadar saparsa? (Büyük sapma → distribution shift, küçük sapma → tutarlı)
2. fig09'da contamination eğrisi keskin tepe veriyor mu, yoksa düz mü? (Düz olması IF'in robust olduğuna işaret)
3. fig07'de Non-IID heterojenliği görsel olarak belirgin mi? (S3'ün motivasyonu için kritik)
